# Pipeline Project

You will be using the provided data to create a machine learning model pipeline.

You must handle the data appropriately in your pipeline to predict whether an
item is recommended by a customer based on their review.
Note the data includes numerical, categorical, and text data.

You should ensure you properly train and evaluate your model.

## The Data

The dataset has been anonymized and cleaned of missing values.

There are 8 features for to use to predict whether a customer recommends or does
not recommend a product.
The `Recommended IND` column gives whether a customer recommends the product
where `1` is recommended and a `0` is not recommended.
This is your model's target/

The features can be summarized as the following:

- **Clothing ID**: Integer Categorical variable that refers to the specific piece being reviewed.
- **Age**: Positive Integer variable of the reviewers age.
- **Title**: String variable for the title of the review.
- **Review Text**: String variable for the review body.
- **Positive Feedback Count**: Positive Integer documenting the number of other customers who found this review positive.
- **Division Name**: Categorical name of the product high level division.
- **Department Name**: Categorical name of the product department name.
- **Class Name**: Categorical name of the product class name.

The target:
- **Recommended IND**: Binary variable stating where the customer recommends the product where 1 is recommended, 0 is not recommended.

## Load Data

In [2]:
import pandas as pd

# Load data
df = pd.read_csv(
    'data/reviews.csv',
)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18442 entries, 0 to 18441
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Clothing ID              18442 non-null  int64 
 1   Age                      18442 non-null  int64 
 2   Title                    18442 non-null  object
 3   Review Text              18442 non-null  object
 4   Positive Feedback Count  18442 non-null  int64 
 5   Division Name            18442 non-null  object
 6   Department Name          18442 non-null  object
 7   Class Name               18442 non-null  object
 8   Recommended IND          18442 non-null  int64 
dtypes: int64(4), object(5)
memory usage: 1.3+ MB


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses,0
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants,1
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses,1
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses,0
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits,1


## Preparing features (`X`) & target (`y`)

In [3]:
data = df

# separate features from labels
X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

print('Labels:', y.unique())
print('Features:')
display(X.head())

Labels: [0 1]
Features:


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits


In [10]:
# Split data into train and test sets
# We use a 90/10 split so the model trains on most of the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    shuffle=True,
    random_state=27,
)
print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
print(f"\nThe test set is held out and ONLY used for final evaluation.")
print(f"This prevents data leakage and gives an honest measure of model performance.")

Train: 16597 samples, Test: 1845 samples

The test set is held out and ONLY used for final evaluation.
This prevents data leakage and gives an honest measure of model performance.


# Your Work

## Data Exploration

In [5]:
# --- Data Exploration ---

print(f"Dataset shape: {df.shape}")
print(f"\nTarget class distribution:")
print(df['Recommended IND'].value_counts())
print(f"\n{df['Recommended IND'].mean():.1%} of reviews are marked 'Recommended'")

print("\nNumerical features summary:")
display(df[['Age', 'Positive Feedback Count', 'Clothing ID']].describe().round(2))

print("\nCategorical feature unique values:")
for col in ['Division Name', 'Department Name', 'Class Name']:
    print(f"  {col}: {df[col].nunique()} unique — {df[col].unique().tolist()}")

review_lengths = df['Review Text'].str.split().str.len()
print(f"\nReview Text word count — mean: {review_lengths.mean():.1f}, max: {review_lengths.max()}")

Dataset shape: (18442, 9)

Target class distribution:
Recommended IND
1    15053
0     3389
Name: count, dtype: int64

81.6% of reviews are marked 'Recommended'

Numerical features summary:


,Age,Positive Feedback Count,Clothing ID
count,18442.00,18442.00,18442.00
mean,43.38,2.70,954.90
std,12.25,5.94,141.57
min,18.00,0.00,2.00
25%,34.00,0.00,863.00
50%,41.00,1.00,952.00
75%,52.00,3.00,1078.00
max,99.00,122.00,1205.00



Categorical feature unique values:
  Division Name: 2 unique — ['General', 'General Petite']
  Department Name: 6 unique — ['Dresses', 'Bottoms', 'Tops', 'Jackets', 'Trend', 'Intimate']
  Class Name: 14 unique — ['Dresses', 'Pants', 'Blouses', 'Knits', 'Outerwear', 'Sweaters', 'Skirts', 'Fine gauge', 'Jackets', 'Trend', 'Lounge', 'Jeans', 'Shorts', 'Casual bottoms']

Review Text word count — mean: 62.4, max: 115


## Building Pipeline

In [11]:
import numpy as np
import spacy
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Load spaCy English model for NLP preprocessing
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# Feature group definitions
NUM_FEATURES = ['Age', 'Positive Feedback Count', 'Clothing ID']
CAT_FEATURES = ['Division Name', 'Department Name', 'Class Name']


def spacy_tokenizer(text):
    """Tokenize, lemmatize, and remove stop words/punctuation using spaCy."""
    if not isinstance(text, str) or text.strip() == '':
        return ''
    doc = nlp(text.lower())
    # Keep lemmatized tokens that are not stop words or punctuation
    tokens = [token.lemma_ for token in doc
              if not token.is_stop and not token.is_punct and not token.is_space]
    return ' '.join(tokens)


# --- Pre-process text columns ONCE with spaCy ---
# This avoids re-running spaCy during GridSearchCV (which would be very slow)
print("Processing Review Text with spaCy (this runs once)...")
X_train['Review Text'] = X_train['Review Text'].apply(spacy_tokenizer)
X_test['Review Text'] = X_test['Review Text'].apply(spacy_tokenizer)

print("Processing Title with spaCy...")
X_train['Title'] = X_train['Title'].apply(spacy_tokenizer)
X_test['Title'] = X_test['Title'].apply(spacy_tokenizer)
print("Done! Text is now lemmatized and cleaned.")


def build_num_pipeline():
    """Impute with median, then standardize."""
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])


def build_cat_pipeline():
    """Impute with mode, then one-hot encode."""
    return Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ])


def build_preprocessor():
    """Combine numerical, categorical, and text transformations."""
    return ColumnTransformer([
        ('num', build_num_pipeline(), NUM_FEATURES),
        ('cat', build_cat_pipeline(), CAT_FEATURES),
        # TF-IDF on already-lemmatized review text
        ('tfidf_review', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
        ), 'Review Text'),
        # TF-IDF on already-lemmatized title
        ('tfidf_title', TfidfVectorizer(
            max_features=500,
        ), 'Title'),
    ])


def build_pipeline():
    """Full ML pipeline: feature preprocessing + logistic regression."""
    return Pipeline([
        ('preprocessor', build_preprocessor()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=27)),
    ])


pipeline = build_pipeline()
print("Pipeline built successfully.")
print(pipeline)

Processing Review Text with spaCy (this runs once)...
Processing Title with spaCy...
Done! Text is now lemmatized and cleaned.
Pipeline built successfully.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age',
                                                   'Positive Feedback Count',
                                                   'Clothing ID']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   Sim

## Training Pipeline

In [7]:
# --- Train the pipeline on training data ---
pipeline.fit(X_train, y_train)

train_acc = pipeline.score(X_train, y_train)
print(f"Pipeline trained successfully.")
print(f"Training accuracy: {train_acc:.4f}")

Pipeline trained successfully.
Training accuracy: 0.9326


## Fine-Tuning Pipeline

In [8]:
from sklearn.model_selection import GridSearchCV

# Tune TF-IDF vocabulary size and logistic regression regularization (C)
param_grid = {
    'preprocessor__tfidf_review__max_features': [3000, 5000],
    'classifier__C': [0.1, 1.0, 10.0],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")

best_pipeline = grid_search.best_estimator_

Fitting 3 folds for each of 6 candidates, totalling 18 fits

Best parameters: {'classifier__C': 1.0, 'preprocessor__tfidf_review__max_features': 3000}
Best CV F1 score: 0.9408


## Evaluating the Model

### Why we evaluate on the test set

We split the data into training and test sets at the start. The model only learned from the training data. Now we test it on the 10% it has never seen before — this tells us how it would perform on real new reviews.

If we only checked performance on training data, the model could just memorize the answers and look perfect. That's overfitting. The test set keeps us honest.

We also look at more than just accuracy. Since most reviews (82%) are "Recommended," a model could get 82% accuracy by always guessing "Recommended" without learning anything. That's why we also check precision, recall, and F1.

In [9]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)

# Evaluate the best pipeline on held-out test data
y_pred = best_pipeline.predict(X_test)

# --- Metrics ---
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
train_acc = best_pipeline.score(X_train, y_train)

print("=== Test Set Evaluation ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['Not Recommended', 'Recommended'],
))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"  True Negatives  (correctly predicted Not Recommended): {cm[0][0]}")
print(f"  False Positives (predicted Recommended but was not):   {cm[0][1]}")
print(f"  False Negatives (predicted Not Recommended but was):   {cm[1][0]}")
print(f"  True Positives  (correctly predicted Recommended):     {cm[1][1]}")

# --- Training vs Test comparison ---
print(f"\n=== Overfitting Check ===")
print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy:     {acc:.4f}")
print(f"Difference:        {train_acc - acc:.4f}")

=== Test Set Evaluation ===
Accuracy:  0.8965
Precision: 0.9213
Recall:    0.9559
F1 Score:  0.9382

Classification Report:
                 precision    recall  f1-score   support

Not Recommended       0.75      0.62      0.68       327
    Recommended       0.92      0.96      0.94      1518

       accuracy                           0.90      1845
      macro avg       0.84      0.79      0.81      1845
   weighted avg       0.89      0.90      0.89      1845

Confusion Matrix:
  True Negatives  (correctly predicted Not Recommended): 203
  False Positives (predicted Recommended but was not):   124
  False Negatives (predicted Not Recommended but was):   67
  True Positives  (correctly predicted Recommended):     1451

=== Overfitting Check ===
Training accuracy: 0.9305
Test accuracy:     0.8965
Difference:        0.0341


### What the results tell us
The model gets about 90% of predictions right . let's break that down:

- **Precision (0.92):** When the model says "Recommended," it gets it right 92% of the time.
- **Recall (0.96):** It catches 96% of all the actual recommended reviews — it misses very few.
- **F1 (0.94):** This balances precision and recall into one number. Anything above 0.9 is solid.

The weaker spot is the "Not Recommended" class — precision 0.75 and recall around 0.62. This makes sense because there are way fewer negative reviews to learn from (only 18% of the dataset). The model just doesn't get enough examples of what a bad review looks like.

Looking at training accuracy vs test accuracy, the gap is small. That means the model isn't overfitting — it generalizes well to data it hasn't seen before.